In [ ]:
import numpy as np
import scipy.signal as ss
import matplotlib.pyplot as plt
import skimage.filters
import multiprocessing as mp
from joblib import Parallel, delayed

In [ ]:
USE_ABC  = True

Lx = 12         # tamanho da sala [m]
Ly = 8
Lt = 0.1        # tempo de simulação total [s]

ca = 200        # velocidade do som no ar [m/s]
beta = 0        # damp

Dx = 50e-3      # discretização em x [m]
Dy = 50e-3      # discretização em y [m]
Dt = 4e-5       # time step [s]
fc = 440        # frequencia central da fonte [Hz]
bw = 0.5        # bw da fonte

Nx = round(Lx / Dx)
Ny = round(Ly / Dy)
Nt = round(Lt / Dt)

print(f'Nx: {Nx}, Ny: {Ny}, Nt: {Nt}')

N_shot_spacing = 20
shot_indices   = np.arange(N_shot_spacing, Nx, N_shot_spacing)
n_shots        = len(shot_indices)
print(f'Numero de shots: {n_shots}  (indices: {shot_indices})')

isy = 0

In [ ]:
c = np.zeros((Ny, Nx)) + ca
c[:Ny//2, :] = 1.5 * ca
c[110:120, 70:170] = 500
c[60:70,  50:70]  = 500
c[60:70, 120:200] = 500


r_true   =  c * Dt / Dx
c2_true  = (r_true) ** 2


CFL = np.max(c) * Dt * (1/Dx + 1/Dy)
print(f'Courant Number: {CFL:.3f}')
if CFL > 2:
    print('Warning: Courant number > 2')

vmodel = c.T    # shape (Nx, Ny)

extent_model  = [0, Lx, Ly, 0]
extent_gather = [0, Lx, Lt, 0]

plt.figure(figsize=(10,5))
plt.title('Campo de Velocidades')
plt.imshow(c, aspect='auto', extent=extent_model)
plt.colorbar(label='Velocidade [m/s]')
plt.xlabel('x [m]'); plt.ylabel('y [m]')
shot_x = shot_indices * Dx
plt.scatter(shot_x[1:], np.zeros_like(shot_x[1:])+0.12,
            c='c', marker='o', s=100, zorder=5, label='Shots')


plt.scatter(shot_x[0], 0.12,
            c='r', marker='o', s=100, zorder=5, label='Shot a ser Analisado')

plt.legend(loc='lower left')
plt.tight_layout()
plt.show()


In [ ]:
t  = np.arange(Nt) * Dt
t0 = 10e-3
S  = ss.gausspulse(t - t0, fc, bw)

In [ ]:
def lap(u):
    tmp = -4 * u.copy()

    # with respect to y
    tmp[:-1, :] += u[1:, :]
    tmp[1:, :] += u[:-1, :]

    # with respect to x
    tmp[:, :-1] += u[:, 1:]
    tmp[:, 1:] += u[:, :-1]
    return tmp

def fd_step(u_2, u_1, c2, r, use_abc=USE_ABC, damp=beta, dt=Dt):

    A = 1 + damp * dt
    u_0 = ((2 + damp * dt) * u_1 - u_2 + c2 * lap(u_1)) / A

    if use_abc:
        u_0[-1, :] = u_1[-2, :] + (r[-1, :] - 1)/(r[-1, :] + 1) * (u_0[-2, :] - u_1[-1, :])
        u_0[:, -1] = u_1[:, -2] + (r[:, -1] - 1)/(r[:, -1] + 1) * (u_0[:, -2] - u_1[:, -1])
        u_0[:,  0] = u_1[:,  1] + (r[:,  0] - 1)/(r[:,  0] + 1) * (u_0[:,  1] - u_1[:,  0])

    return u_0


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams['animation.embed_limit'] = 500.0

isx = shot_indices[0]
u_0 = np.zeros((Ny, Nx))
u_1 = np.zeros((Ny, Nx))

u_sim = np.zeros((Ny, Nx, Nt))
surface_record = np.zeros((Nx, Nt))

for nt in range(Nt):
    u_2, u_1   = u_1, u_0
    u_0 = fd_step(u_2, u_1, c2_true, r_true)
    u_0[isy, isx] += S[nt]

    surface_record[:, nt] = u_1[isy, :]

    u_sim[:, :, nt] = u_1

# visualização
vmm = 0.5
fig, ax = plt.subplots(figsize=(10,5))
im = ax.imshow(u_1, extent=[0, Lx, Ly, 0], aspect='auto', vmin=-vmm, vmax=vmm, cmap='bwr')
steps_per_frame = 10
num_frames = Nt // steps_per_frame

fig.suptitle("Simulação da Aquisição de Dados", fontsize=14, fontweight='bold')

def update(frame):
    nt = frame * steps_per_frame
    u_1 = u_sim[:, :, nt]

    im.set_data(u_1)

    E = np.sum(u_1 ** 2)
    ax.set_title(f'Nt = {nt}, E = {E:.2f}')

    return [im, ax.title]

anim = FuncAnimation(fig, update, frames=num_frames, interval=50, blit=False)

plt.close(fig)

HTML(anim.to_jshtml())

In [ ]:
vmodel_smooth = skimage.filters.gaussian(vmodel, sigma=9)
c_smooth      = vmodel_smooth.T
c2_smooth     = (c_smooth * Dt / Dx) ** 2
r_smooth      =  c_smooth * Dt / Dx

v_surface = vmodel[:, 0]
x_array   = np.arange(Nx) * Dx

plt.figure(figsize=(10,5))
plt.title('Campo de Velocidades "Estimado"')
plt.imshow(c_smooth, aspect='auto', extent=extent_model)
plt.colorbar(label='Velo [m/s]')
plt.xlabel('x [m]'); plt.ylabel('y [m]')
plt.tight_layout()
plt.show()

In [ ]:
plt.imshow(surface_record.T, cmap = 'bwr', vmin = -0.2, vmax = 0.2, interpolation='bilinear', aspect='auto')
plt.title("Sinal Recebido pelo eixo x (reta y = 0)")

In [ ]:
u_0 = np.zeros((Ny, Nx))
u_1 = np.zeros((Ny, Nx))

u_up = np.zeros((Ny, Nx, Nt))

gather_for_rtm = surface_record

for nt in range(Nt):
    u_2, u_1 = u_1, u_0
    u_0 = fd_step(u_2, u_1, c2_smooth, r_smooth)
    u_0[isy, :] += gather_for_rtm[:, (Nt - 1)- nt]

    u_up[:, :, (Nt - 1)- nt] = u_1

vmm = 1
fig, ax = plt.subplots(figsize=(10,5))
fig.suptitle("Ondas Ascendentes (Em Tempo Reverso)", fontsize=14, fontweight='bold')

initial_idx = Nt - 1
im = ax.imshow(u_up[:, :, initial_idx], extent=[0, Lx, Ly, 0], aspect='auto', vmin=-vmm, vmax=vmm, cmap='bwr')

steps_per_frame = 10
num_frames = Nt // steps_per_frame

def update(frame):
    nt = frame * steps_per_frame
    plot_idx = nt
    current_wavefield = u_up[:, :, plot_idx]

    im.set_data(current_wavefield)

    E = np.sum(current_wavefield ** 2)

    ax.set_title(f'Nt = {nt} | E = {E:.2f}')

    return [im, ax.title]

# Generate and display the animation
anim = FuncAnimation(fig, update, frames=num_frames, interval=50, blit=False)
plt.close('all')

HTML(anim.to_jshtml())

In [ ]:
u_0 = np.zeros((Ny, Nx))
u_1 = np.zeros((Ny, Nx))

u_down = np.zeros((Ny, Nx, Nt))

for nt in range(Nt):
    u_2, u_1   = u_1, u_0
    u_0 = fd_step(u_2, u_1, c2_smooth, r_smooth)
    u_0[isy, isx] += S[nt]

    surface_record[:, nt] = u_1[isy, :]

    u_down[:, :, nt] = u_1

vmm = 1
fig, ax = plt.subplots(figsize=(10,5))

fig.suptitle("Ondas Descendentes", fontsize=14, fontweight='bold')

im = ax.imshow(u_1, extent=[0, Lx, Ly, 0], aspect='auto', vmin=-vmm, vmax=vmm, cmap='bwr')
steps_per_frame = 10
num_frames = Nt // steps_per_frame

def update(frame):
    nt = frame * steps_per_frame

    u_1 = u_down[:, :, nt]
    im.set_data(u_1)

    E = np.sum(u_1 ** 2)
    ax.set_title(f'Nt = {nt} | E = {E:.2f}')

    return [im, ax.title]

# Generate and display the animation
anim = FuncAnimation(fig, update, frames=num_frames, interval=50, blit=False)
plt.close(fig)

HTML(anim.to_jshtml())

In [ ]:
IC = np.sum(u_up * u_down[:, :, :], axis=2)
vmax = np.percentile(np.abs(IC), 98)

plt.figure(figsize=(12, 12))
plt.subplot(121)
plt.imshow(IC, extent=extent_model, cmap='bwr',
        interpolation='bilinear', vmin=-vmax, vmax=vmax)
plt.title('Condição de Imageamento dado o primeiro Shot')
plt.subplot(122)
plt.title('Mapa de Refletores')
plt.imshow(np.diff(c, axis=0), extent=extent_model, cmap='bwr')

In [ ]:
def process_shot(isx):

    u_0 = np.zeros((Ny, Nx))
    u_1 = np.zeros((Ny, Nx))
    surface_record = np.zeros((Nx, Nt))


    # simulando aquisição fixado a posição da fonte
    for k in range(Nt):
        u_2, u_1   = u_1, u_0
        u_0 = fd_step(u_2, u_1, c2_true, r_true)
        u_0[isy, isx] += S[k]
        surface_record[:, k] = u_1[isy, :]

    gather_for_rtm = surface_record

    # ondas descendentes
    u_0 = np.zeros((Ny, Nx))
    u_1 = np.zeros((Ny, Nx))
    u_down = np.zeros((Ny, Nx, Nt))

    for k in range(Nt):
        u_2, u_1  = u_1, u_0
        u_0 = fd_step(u_2, u_1, c2_smooth, r_smooth)
        u_0[isy, isx] += S[k]
        u_down[:, :, k]  = u_1

    # ondas ascendentes
    u_0 = np.zeros((Ny, Nx))
    u_1 = np.zeros((Ny, Nx))
    u_up   = np.zeros((Ny, Nx, Nt))

    for k in range(Nt):
        u_2, u_1  = u_1, u_0
        u_0          = fd_step(u_2, u_1, c2_smooth, r_smooth)
        u_0[isy, :] += gather_for_rtm[:, -(k + 1)]
        u_up[:, :, Nt - 1 - k]   = u_1

    shot_image = np.sum(u_up * u_down[:, :, :], axis=2)

    print(f'  Shot isx={isx} done.')
    return isx, shot_image, gather_for_rtm

n_workers = 3

results = Parallel(n_jobs=n_workers, backend='loky')(
    delayed(process_shot)(isx) for isx in shot_indices
)

results.sort(key=lambda r: r[0])

In [ ]:
stack = np.zeros((Ny, Nx))

for shot_no, (isx, shot_image, gather_for_rtm) in enumerate(results):
    stack += shot_image
    if shot_no % 1 == 0:
        fig, axs = plt.subplots(1, 3, figsize=(18, 4))
        fig.suptitle(f'Shot {shot_no+1}/{n_shots}  —  isx={isx}  (x={isx*Dx:.2f} m)')
        axs[0].imshow(gather_for_rtm.T, extent=extent_gather, cmap='bwr',
                      vmin=-0.1, vmax=0.1, interpolation='bilinear', aspect='auto')

        axs[0].set_xlabel('x [m]'); axs[0].set_ylabel('Time [s]')

        axs[1].set_title('Shot image')
        vmax1 = np.percentile(np.abs(shot_image), 98) or 1
        axs[1].imshow(shot_image, extent=extent_model, cmap='bwr',
                      vmin=-vmax1, vmax=vmax1, interpolation='bilinear', aspect='auto')
        axs[1].set_xlabel('x [m]'); axs[1].set_ylabel('y [m]')

        axs[2].set_title(f'Stack {shot_no+1}/{n_shots}')
        vmax2 = np.percentile(np.abs(stack), 98) or 1
        axs[2].imshow(stack, extent=extent_model, cmap='bwr',
                      vmin=-vmax2, vmax=vmax2, interpolation='bilinear', aspect='auto')
        axs[2].set_xlabel('x [m]'); axs[2].set_ylabel('y [m]')

        plt.tight_layout()
        plt.show()


In [ ]:
vmax = np.percentile(np.abs(stack), 98)

plt.figure(figsize=(12, 12))
plt.subplot(121)
plt.imshow(stack, extent=extent_model, cmap='bwr',
        interpolation='bilinear', vmin=-vmax, vmax=vmax)
plt.title('Imagem Final')
plt.subplot(122)
plt.title('Mapa de Refletores')
plt.imshow(np.diff(c, axis=0), extent=extent_model, cmap='bwr',)